# From GPT to SmolVLA — Part 2: building the action expert from scratch

In **Part 1** you built a decoder-only GPT: embed tokens → stack of *causal self-attention* + FFN
blocks → project to vocab logits → cross-entropy → autoregressive `generate()`. In the original
Transformer figure, that is the **decoder stack with the cross-attention removed**.

**SmolVLA's action expert is that same decoder stack with the cross-attention put *back*** — plus a
different output head and a different training objective. This notebook builds exactly that delta, in
the same from-scratch, one-idea-at-a-time style, and trains it on a toy robot task so every piece runs
on a CPU in seconds. (Faithful to `SmolVLA.pdf` §3.1.)

### The three deltas from your GPT
| | GPT (Part 1) | SmolVLA action expert (here) |
|---|---|---|
| **Output** | discrete token, softmax over vocab | **continuous action chunk** `A = (a_t,…,a_{t+n})` |
| **Objective** | cross-entropy (classification) | **flow matching** — regress a velocity field (§3.1) |
| **Attention** | causal self-attention only | **interleaved cross-attn (CA) + causal self-attn (SA)** |
| **Conditioning** | previous characters | **VLM features** `o_t` (image + language + state token) |

We tackle them in that order: first *what the model outputs* (continuous actions + flow matching),
then *how it reads the world* (cross-attention), then we assemble and train the whole expert.


## The big picture — SmolVLA drawn as the original Transformer figure

Before the code, here is where everything below fits. This is **SmolVLA in the exact layout of the
"Attention Is All You Need" Figure 1**, using that paper's color legend. Read each stack bottom-to-top.
The **encoder** became a multimodal **VLM backbone**; the **decoder** became the **flow-matching action
expert** we build in this notebook.

<div style="overflow-x:auto">

<svg viewBox="0 0 1080 1230" width="100%" style="max-width:780px;height:auto;display:block;margin:auto" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="SmolVLA in the style of the original Transformer Figure 1">
<rect x="0" y="0" width="1080" height="1230" rx="16" fill="#FBFAF6"/>
<defs><marker id="ah" viewBox="0 0 8 8" refX="6.5" refY="4" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M0,0 L7,4 L0,8 z" fill="#4A463E"/></marker></defs>
<path d="M174,1103 V1088 H300 V1053" fill="none" stroke="#4A463E" stroke-width="1.8"/>
<path d="M300,1103 V1053" fill="none" stroke="#4A463E" stroke-width="1.8"/>
<path d="M426,1103 V1088 H300" fill="none" stroke="#4A463E" stroke-width="1.8"/>
<path d="M300,1053 V727" fill="none" stroke="#4A463E" stroke-width="1.8" marker-end="url(#ah)"/>
<path d="M780,1103 V443" fill="none" stroke="#4A463E" stroke-width="1.8" marker-end="url(#ah)"/>
<circle cx="300" cy="1050" r="8" fill="#FBFAF6" stroke="#4A463E" stroke-width="1.4"/><text x="300" y="1054" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">+</text>
<text x="348" y="1054" text-anchor="start" font-size="10.5" fill="#6b6456" font-family="Georgia,Times New Roman,serif">pos. enc.</text>
<circle cx="780" cy="1078" r="8" fill="#FBFAF6" stroke="#4A463E" stroke-width="1.4"/><text x="780" y="1082" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">+</text>
<text x="828" y="1082" text-anchor="start" font-size="10.5" fill="#6b6456" font-family="Georgia,Times New Roman,serif">pos. + &#964;</text>
<rect x="120" y="758" width="360" height="288" rx="12" fill="none" stroke="#B4AEA1" stroke-width="1.3" stroke-dasharray="5 5"/>
<rect x="600" y="618" width="360" height="428" rx="12" fill="none" stroke="#B4AEA1" stroke-width="1.3" stroke-dasharray="5 5"/>
<text x="300" y="750" text-anchor="middle" font-size="12.5" fill="#7C7566" font-style="italic" font-family="Georgia,Times New Roman,serif">SmolLM2 decoder &#183; VLM backbone (SmolVLM-2)</text>
<text x="780" y="610" text-anchor="middle" font-size="12.5" fill="#7C7566" font-style="italic" font-family="Georgia,Times New Roman,serif">flow-matching action expert  v&#952;  (hidden 0.75&#183;d)</text>
<path d="M112,766 h-8 V1038 h8" fill="none" stroke="#B4AEA1" stroke-width="1.3"/>
<text x="70" y="906" transform="rotate(-90 70 906)" text-anchor="middle" font-size="13" fill="#6b6456" font-style="italic" font-family="Georgia,Times New Roman,serif">N = L/2 &#215;</text>
<path d="M968,626 h8 V1038 h-8" fill="none" stroke="#B4AEA1" stroke-width="1.3"/>
<text x="1006" y="832" transform="rotate(90 1006 832)" text-anchor="middle" font-size="13" fill="#6b6456" font-style="italic" font-family="Georgia,Times New Roman,serif">interleaved CA / SA &#215; N</text>
<rect x="120" y="1103" width="108" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="174" y="1122" text-anchor="middle" font-size="12.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">SigLIP +</text><text x="174" y="1138" text-anchor="middle" font-size="12.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">pixel-shuffle</text>
<rect x="246" y="1103" width="108" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="300" y="1130" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Tokenizer</text>
<rect x="372" y="1103" width="108" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="426" y="1122" text-anchor="middle" font-size="12.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Linear &#8594;</text><text x="426" y="1138" text-anchor="middle" font-size="12.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">state token</text>
<text x="174" y="1170" text-anchor="middle" font-size="12.5" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Camera images</text><text x="174" y="1185" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">&#8594; 64 visual tokens</text>
<text x="300" y="1170" text-anchor="middle" font-size="12.5" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Language instruction</text>
<text x="426" y="1170" text-anchor="middle" font-size="12.5" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Robot state</text><text x="426" y="1185" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">&#8594; 1 token</text>
<rect x="150" y="979" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="300" y="1005" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Multi-Head Self-Attention</text>
<rect x="150" y="909" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="300" y="935" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="150" y="839" width="300" height="42" rx="8" fill="#C6DCEF" stroke="#7AA6C9" stroke-width="1.4"/><text x="300" y="865" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Feed Forward</text>
<rect x="150" y="769" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="300" y="795" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="150" y="683" width="300" height="44" rx="8" fill="#E7E3DA" stroke="#B4AEA1" stroke-width="1.4"/><text x="300" y="702" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">VLM features  o&#8348;</text><text x="300" y="718" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">taken at layer N = L/2 (layer skip)</text>
<rect x="630" y="1103" width="300" height="46" rx="8" fill="#F6CFC9" stroke="#CE8E86" stroke-width="1.4"/><text x="780" y="1122" text-anchor="middle" font-size="13" fill="#33302A" font-family="Georgia,Times New Roman,serif">Linear proj + position + &#964;-embed</text><text x="780" y="1138" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">(the noised action chunk becomes tokens)</text>
<text x="780" y="1170" text-anchor="middle" font-size="12.5" fill="#33302A" font-style="italic" font-family="Georgia,Times New Roman,serif">Noisy action chunk  A&#7511;  + flow time &#964;</text>
<rect x="630" y="979" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="780" y="1005" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Masked (causal) Self-Attention</text>
<rect x="630" y="909" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="935" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="630" y="839" width="300" height="42" rx="8" fill="#F8C98C" stroke="#CF9445" stroke-width="1.4"/><text x="780" y="865" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Multi-Head Cross-Attention</text>
<rect x="630" y="769" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="795" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="630" y="699" width="300" height="42" rx="8" fill="#C6DCEF" stroke="#7AA6C9" stroke-width="1.4"/><text x="780" y="725" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Feed Forward</text>
<rect x="630" y="629" width="300" height="42" rx="8" fill="#FBF0A6" stroke="#CBB85A" stroke-width="1.4"/><text x="780" y="655" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Add &amp; Norm</text>
<rect x="655" y="539" width="250" height="42" rx="8" fill="#D9CBE9" stroke="#A08CC0" stroke-width="1.4"/><text x="780" y="565" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Linear</text>
<rect x="645" y="464" width="270" height="52" rx="8" fill="#C8E6C0" stroke="#77AE6C" stroke-width="1.4"/><text x="780" y="486" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Flow matching</text><text x="780" y="503" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">integrate v&#952; over &#964;: 0 &#8594; 1</text>
<rect x="655" y="397" width="250" height="46" rx="8" fill="#E7E3DA" stroke="#B4AEA1" stroke-width="1.4"/><text x="780" y="416" text-anchor="middle" font-size="14.5" fill="#33302A" font-family="Georgia,Times New Roman,serif">Action chunk  A&#8348;</text><text x="780" y="432" text-anchor="middle" font-size="11" fill="#6b6456" font-family="Georgia,Times New Roman,serif">(a&#8348;, &#8230;, a&#8348;&#8330;&#8345;)</text>
<text x="780" y="378" text-anchor="middle" font-size="12.5" fill="#7C7566" font-style="italic" font-family="Georgia,Times New Roman,serif">Output: continuous actions</text>
<path d="M450,699 H556 V858 H628" fill="none" stroke="#B5502E" stroke-width="1.8" marker-end="url(#ah)"/>
<path d="M450,714 H572 V872 H628" fill="none" stroke="#B5502E" stroke-width="1.8" marker-end="url(#ah)"/>
<text x="512" y="690" text-anchor="middle" font-size="11.5" fill="#B5502E" font-style="italic" font-family="Georgia,Times New Roman,serif">features o&#8348;</text>
<text x="512" y="758" text-anchor="middle" font-size="11.5" fill="#B5502E" font-style="italic" font-family="Georgia,Times New Roman,serif">&#8594; keys, values</text>
</svg>

</div>

*Colors follow the original legend: pink = embedding/projection, orange = attention, yellow = Add&nbsp;&amp;&nbsp;Norm,
blue = feed-forward, purple = Linear, green = output (softmax&nbsp;&#8594;&nbsp;flow matching).*

**The three swaps from a vanilla Transformer / your Part&nbsp;1 GPT:**
1. **Encoder &#8594; VLM backbone.** Images (`SigLIP` + pixel-shuffle &#8594; 64 tokens) + language (tokenized)
   + robot state (&#8594; 1 token) are concatenated through `SmolLM2`; features are read at layer **N&nbsp;=&nbsp;L/2**.
2. **Softmax &#8594; flow matching.** The green output block regresses a velocity field `v&#952;` and
   **integrates** it from noise to actions, instead of a softmax over a vocabulary.
3. **Cross-attention returns.** Your GPT was the decoder *without* the middle attention box; SmolVLA puts
   it back so action tokens can read the VLM features `o&#8348;` (the orange arrow), while causal
   self-attention keeps each action from seeing future actions.

> One simplification: the original draws SA **and** CA inside one decoder block. SmolVLA **interleaves**
> them &mdash; each block is *either* a CA *or* an SA layer (the `&#215;N` bracket). We show both here for a
> clean 1-to-1 map, and the code in &sect;4 does the real interleaving via `layer_types`.


In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", device)

## 1. Delta 1 — the model outputs a *continuous action chunk*, not a token

A language model predicts **one discrete symbol** at a time from a fixed vocabulary, so its head is a
`Linear -> softmax` and its loss is cross-entropy. A robot policy instead predicts **real-valued
actions** (joint velocities, gripper position, …), and it predicts a whole **chunk** of the next `n`
timesteps at once:

$$A_t = (a_t, a_{t+1}, \dots, a_{t+n}), \qquad a \in \mathbb{R}^{\text{action\_dim}}.$$

Predicting a chunk (SmolVLA, π0, ACT all do this) means one forward pass yields `n` actions the robot
can execute open-loop — crucial for the **asynchronous inference** used at the edge (§3.3): the robot
keeps acting from its queue while the next chunk is computed.

So the output shape changes from `(B, T, vocab_size)` logits to `(B, n, action_dim)` real numbers, and
the natural loss becomes **regression** (mean-squared error), not classification. But naively
regressing the mean action is bad — action distributions are *multimodal* (many valid ways to reach a
goal), and MSE-to-the-mean blurs them. That is the problem **flow matching** solves next.

## 2. Delta 2 — flow matching: how to *generate* continuous actions

Flow matching (Lipman et al. 2022; Liu 2022; used by π0 and SmolVLA) turns generation into a simple
regression. The idea:

1. Put pure noise `ε ~ N(0, I)` at time `τ=0` and the true action chunk `A` at time `τ=1`, and draw a
   **straight line** between them:
   $$A^\tau = (1-\tau)\,\varepsilon + \tau\,A.$$
2. The constant velocity that carries noise to data along that line is just
   $$v = \frac{d A^\tau}{d\tau} = A - \varepsilon.$$
3. **Train a network `v_θ(A^τ, τ, o)` to predict that velocity**, conditioned on the VLM features
   `o`. That is a plain MSE regression — no softmax, no vocabulary.
4. **Sample** by starting at noise `ε` and integrating the learned velocity field from `τ=0` to `τ=1`
   (a few Euler steps). Because `v_θ` depends on `A^τ`, it can bend toward *whichever* mode the noise
   sample is near — that is how it captures multimodal action distributions.

> **Sign convention note.** SmolVLA's paper (§3.1) writes the field as `u = ε − A` with
> `A^τ = τA + (1−τ)ε`. That is the same construction with the opposite sign / integration direction.
> We use `v = A − ε` and integrate `τ: 0→1` (noise→data) because it is the most direct to read.

### A toy "robot": reach a 2-D goal
To keep everything runnable, our "world observation" is a 2-D **goal**, and the ground-truth action
chunk is the straight-line trajectory from the origin to that goal over `n` steps. A *fixed random*
network stands in for the pretrained **VLM**: it turns the goal into a sequence of feature tokens
`o` (shape `(B, S, feat_dim)`) — exactly the interface SmolVLM-2 exposes to the action expert.

In [ ]:
ACTION_DIM, CHUNK_LEN = 2, 8          # 2-D actions, chunks of 8 timesteps
FEAT_TOKENS, FEAT_DIM = 16, 32        # the fake "VLM" emits 16 feature tokens of width 32

# A FIXED random "VLM": goal -> sequence of feature tokens. Not trained (stands in for SmolVLM-2).
Wg = torch.randn(2, 64, device=device) / 2
Wf = torch.randn(64, FEAT_TOKENS * FEAT_DIM, device=device) / 8

def make_batch(B):
    goal = (torch.rand(B, 2, device=device) * 4 - 2)          # goal in [-2, 2]^2  == the "observation"
    h = torch.tanh(goal @ Wg)                                 # (B, 64)
    feats = (h @ Wf).view(B, FEAT_TOKENS, FEAT_DIM)           # (B, S, feat_dim) -- fake VLM features o
    t = torch.arange(1, CHUNK_LEN + 1, device=device).float() / CHUNK_LEN
    A = goal[:, None, :] * t[None, :, None]                   # (B, n, action_dim) straight-line trajectory
    return goal, feats, A

goal, feats, A = make_batch(4)
print("goal (observation):", tuple(goal.shape))
print("VLM features o    :", tuple(feats.shape))
print("action chunk A    :", tuple(A.shape), "  <- (batch, chunk_len, action_dim)")
print("\nfirst trajectory (should march from ~0 toward the goal", [round(v,2) for v in goal[0].tolist()], "):")
print(A[0].round(decimals=2))

## 3. Delta 3 — one attention block that does **both** cross- and causal self-attention

Recall Part 1's `Head`: `Q·Kᵀ → scale → mask → softmax → ·V`, with Q, K, V all coming from the *same*
sequence. The only thing that distinguishes the three attention flavors in the original Transformer is
**where Q, K, V come from** and **whether the future is masked**. So a single class covers both cases
we need:

- **Cross-attention (CA)**: `Q` from the **action tokens**, `K`,`V` from the **VLM features** `o`. No
  causal mask — an action token may read the entire observation. *This is the encoder→decoder arrow
  from the Transformer figure.*
- **Causal self-attention (SA)**: `Q`,`K`,`V` all from the action tokens, with a causal mask so action
  token `t` sees only actions `≤ t` in the chunk (SmolVLA masks SA so there are no future-action
  dependencies — §3.1).

`ctx=None` ⇒ self-attention; passing `ctx` ⇒ cross-attention. That's the whole difference.

In [ ]:
class MHA(nn.Module):
    """Multi-head attention. Self-attention if ctx is None, else cross-attention onto ctx.
    causal=True masks the future (used for the SA layers on action tokens)."""
    def __init__(self, d, n_heads, causal=False):
        super().__init__()
        assert d % n_heads == 0
        self.h, self.hd, self.causal = n_heads, d // n_heads, causal
        self.q = nn.Linear(d, d, bias=False)   # query projection (from action tokens)
        self.k = nn.Linear(d, d, bias=False)   # key   projection (from ctx, or self)
        self.v = nn.Linear(d, d, bias=False)   # value projection (from ctx, or self)
        self.proj = nn.Linear(d, d)            # mix heads back together

    def forward(self, x, ctx=None):
        ctx = x if ctx is None else ctx        # self-attention <=> keys/values are the queries
        B, Tq, d = x.shape; Tk = ctx.shape[1]
        # split into heads: (B, T, d) -> (B, n_heads, T, head_dim)
        q = self.q(x).view(B, Tq, self.h, self.hd).transpose(1, 2)
        k = self.k(ctx).view(B, Tk, self.h, self.hd).transpose(1, 2)
        v = self.v(ctx).view(B, Tk, self.h, self.hd).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * self.hd ** -0.5      # (B, h, Tq, Tk) scaled scores
        if self.causal:                                        # forbid attending to future actions
            m = torch.tril(torch.ones(Tq, Tk, device=x.device)).bool()
            att = att.masked_fill(~m, float("-inf"))
        att = att.softmax(-1)
        out = (att @ v).transpose(1, 2).contiguous().view(B, Tq, d)
        return self.proj(out)


class FeedForward(nn.Module):
    """Per-token MLP, same role as in Part 1 (expand -> nonlinearity -> contract)."""
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x): return self.net(x)

## 4. Assemble the action expert — interleaved CA / SA blocks

Now stack it like Part 1's GPT, with three changes:

1. **Input**: instead of embedding token *ids*, we linearly project the (noisy) action chunk `A^τ` into
   token vectors, add a learned **position** embedding over the chunk, and add a **time embedding** of
   `τ` (the network must know how far along the noise→data path it is).
2. **Interleaving**: SmolVLA makes each block *either* a CA layer *or* an SA layer (not both), e.g.
   `[CA, SA, CA, SA]`. CA layers read the VLM features; SA layers are causal over the chunk. We keep
   the Part 1 recipe of **pre-norm + residual** around each sublayer.
3. **Output head**: a `Linear` back to `action_dim` produces the predicted **velocity** `v_θ` for every
   action token — not vocab logits.

In [ ]:
class ActionExpert(nn.Module):
    def __init__(self, d=64, n_heads=4, layer_types=("ca", "sa", "ca", "sa")):
        super().__init__()
        self.in_proj   = nn.Linear(ACTION_DIM, d)     # noisy action chunk A^tau -> tokens
        self.pos       = nn.Embedding(CHUNK_LEN, d)    # where in the chunk each action sits
        self.tau_embed = nn.Linear(1, d)               # flow-matching time conditioning
        self.feat_proj = nn.Linear(FEAT_DIM, d)        # adapt VLM features to the expert width
        self.types = layer_types
        # one attention sublayer per block: causal SA, or CA onto the features
        self.attn = nn.ModuleList([MHA(d, n_heads, causal=(t == "sa")) for t in layer_types])
        self.ln1  = nn.ModuleList([nn.LayerNorm(d) for _ in layer_types])
        self.ln2  = nn.ModuleList([nn.LayerNorm(d) for _ in layer_types])
        self.ff   = nn.ModuleList([FeedForward(d) for _ in layer_types])
        self.ln_f = nn.LayerNorm(d)
        self.out  = nn.Linear(d, ACTION_DIM)           # -> predicted velocity field v_theta

    def forward(self, a_tau, tau, feats):
        # a_tau: (B, n, action_dim) noisy actions | tau: (B, 1) time | feats: (B, S, feat_dim)
        B, n, _ = a_tau.shape
        pos = self.pos(torch.arange(n, device=a_tau.device))
        x = self.in_proj(a_tau) + pos[None] + self.tau_embed(tau)[:, None, :]
        f = self.feat_proj(feats)                      # adapted VLM features (keys/values for CA)
        for i, t in enumerate(self.types):
            ctx = f if t == "ca" else None             # CA reads features; SA reads the chunk (causal)
            x = x + self.attn[i](self.ln1[i](x), ctx)  # pre-norm + residual (as in Part 1)
            x = x + self.ff[i](self.ln2[i](x))
        return self.out(self.ln_f(x))                  # (B, n, action_dim) velocity per action token

model = ActionExpert().to(device)
print("action-expert parameters:", sum(p.numel() for p in model.parameters()) / 1e3, "K")

## 5. Train with the flow-matching objective

The training step is a direct transcription of §2, and mirrors SmolVLA's loss
$\mathcal{L}(\theta)=\mathbb{E}\big[\lVert v_\theta(A^\tau, o) - (A-\varepsilon)\rVert^2\big]$:

1. sample a real chunk `A` and its features `o`,
2. sample noise `ε` and a random time `τ ∈ [0,1]`,
3. form the point on the line `A^τ = (1−τ)ε + τA`,
4. regress the network's output onto the true velocity `A − ε`.

No `generate()`, no sampling loop during training — just one MSE per step.

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for it in range(3001):
    goal, feats, A = make_batch(256)
    eps = torch.randn_like(A)                                  # noise at tau=0
    tau = torch.rand(A.size(0), 1, device=device)             # random time in [0,1]
    a_tau  = (1 - tau[..., None]) * eps + tau[..., None] * A   # point on the noise->data line
    target = A - eps                                           # the velocity that line travels at
    pred   = model(a_tau, tau, feats)                          # v_theta(A^tau, tau, o)
    loss   = F.mse_loss(pred, target)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if it % 500 == 0:
        print(f"step {it:4d} | flow-matching loss {loss.item():.4f}")

## 6. Sample — integrate the ODE from noise to an action chunk

Inference is the flow-matching analogue of GPT's `generate()`. Start from pure noise and take a few
Euler steps along the learned velocity field:

$$x \leftarrow x + v_\theta(x, \tau, o)\,\Delta\tau, \qquad \tau: 0 \to 1.$$

With ~20 steps the endpoint of the sampled chunk should land on the goal encoded in the VLM features —
i.e. the expert has learned to *read the observation via cross-attention* and *emit a valid trajectory
via flow matching*.

In [ ]:
@torch.no_grad()
def sample_chunk(feats, steps=20):
    """Generate an action chunk by integrating the velocity field from noise (tau=0) to data (tau=1)."""
    B = feats.size(0)
    x = torch.randn(B, CHUNK_LEN, ACTION_DIM, device=device)   # start at pure noise
    for i in range(steps):
        tau = torch.full((B, 1), i / steps, device=device)
        x = x + model(x, tau, feats) * (1.0 / steps)           # Euler step along v_theta
    return x

goal, feats, A = make_batch(512)
pred = sample_chunk(feats)
endpoint_err = (pred[:, -1, :] - goal).norm(dim=-1).mean().item()
print(f"mean endpoint error (sampled final action vs goal): {endpoint_err:.3f}")
for b in range(3):
    print(f"  goal {[round(v,2) for v in goal[b].tolist()]}  ->  "
          f"sampled endpoint {[round(v,2) for v in pred[b, -1].tolist()]}")

### Optional: see the trajectories
If `matplotlib` is installed, this plots a few sampled action chunks (lines) against their goals
(stars). They should trace roughly straight paths from the origin to each goal.

In [ ]:
try:
    import matplotlib.pyplot as plt
    g, f, _ = make_batch(6)
    traj = sample_chunk(f).cpu()
    plt.figure(figsize=(5, 5))
    for b in range(6):
        xy = torch.cat([torch.zeros(1, 2), traj[b]], dim=0)   # prepend the origin
        line, = plt.plot(xy[:, 0], xy[:, 1], "-o", ms=3)
        plt.plot(*g[b].cpu(), "*", ms=16, color=line.get_color())
    plt.title("sampled action chunks (lines) vs goals (stars)")
    plt.axhline(0, lw=.5, c="gray"); plt.axvline(0, lw=.5, c="gray"); plt.gca().set_aspect("equal")
    plt.show()
except ImportError:
    print("matplotlib not installed - skipping the plot (the endpoint errors above already show it works)")

## 7. From this toy to real SmolVLA — and to this repo

You just built the mechanism at the heart of SmolVLA's action expert. Here is what the real system
adds on top, and where each piece lives in `SmolVLA.pdf`:

- **Real VLM features (§3.1).** Our fixed random `make_batch` stands in for **SmolVLM-2**: a SigLIP
  vision encoder + SmolLM2 decoder. Images are pixel-shuffled to **64 visual tokens**, the language
  instruction is tokenized, and the robot **state** is projected to a **single token** by a linear
  layer; all are concatenated and run through the decoder. The action expert cross-attends to the
  resulting features — exactly our `feats`.
- **Layer skipping (§3.1).** SmolVLA conditions on features from layer **N = L/2**, not the last layer,
  halving both the VLM and the expert's compute. A speed knob, not a new mechanism.
- **Skinny expert (§3.1).** The expert uses hidden size **0.75×d** of the VLM, for cheaper inference.
- **Beta-sampled time.** SmolVLA samples `τ` from a **Beta** distribution (not uniform) to focus
  training near the data end of the path. Try swapping `torch.rand` for a Beta sample below.
- **Asynchronous inference (§3.3).** Because the expert emits a *chunk*, a `RobotClient` executes from a
  queue while a (possibly remote) `PolicyServer` computes the next chunk — the key to running on
  low-power edge hardware. That is precisely what this repo (`openspec/changes/smolvla-edge-deployment/`)
  is about: fine-tuning, on-device inference, the client/server split, and latency benchmarking.

### Exercises
1. **Multimodality.** Make each goal reachable by *two* trajectories (e.g. clockwise vs counter-clockwise
   arc) and confirm flow-matching samples pick one mode per run, instead of averaging to the middle.
2. **Beta time.** Replace `tau = torch.rand(...)` with a `Beta(1.5, 1.0)` sample and see if fewer Euler
   steps then suffice at sampling time.
3. **Ablate cross-attention.** Set `layer_types=("sa","sa","sa","sa")` (no CA). The expert can no longer
   read the goal — watch the endpoint error blow up. This isolates *what cross-attention buys you*.
4. **Chunk causality.** Print an SA layer's attention weights and confirm they are lower-triangular over
   the chunk (no action attends to a future action), just like Part 1.
5. **Fewer Euler steps.** Drop `steps` in `sample_chunk` from 20 → 5 → 2 and watch endpoint error vs
   inference cost — the trade-off that matters on the edge.
